# RQ2 Step By Step

Provides a step-by-step reconstruction of the RQ2 regional predictability analysis.

**Provenance.** This is an English-facing execution copy of the original project notebook. The computational cells are preserved so that the analysis remains reproducible. Chinese narrative cells have been replaced by concise English stage notes; the original notebook remains unchanged in the source project.

## Analysis stage 1

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 2

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
from pathlib import Path 
import warnings 
import numpy as np 
import pandas as pd 
import matplotlib .pyplot as plt 
from IPython .display import display ,Image ,Markdown 

# Implementation note translated in the organised English notebook.
candidates =[Path .cwd (),*Path .cwd ().parents ]
PROJECT_ROOT =next ((p for p in candidates if (p /'RQ2').exists ()and (p /'data'/'processed').exists ()),None )
if PROJECT_ROOT is None :
    raise FileNotFoundError ('\u627e\u4e0d\u5230 data3001-project \u6839\u76ee\u5f55。\u8bf7\u4ece\u9879\u76ee\u6839\u76ee\u5f55\u6216 RQ2 \u76ee\u5f55\u8fd0\u884c。')

RQ2 =PROJECT_ROOT /'RQ2'
DATA =RQ2 /'data'
FIG =RQ2 /'figures'
PROCESSED =PROJECT_ROOT /'data'/'processed'

plt .style .use ('seaborn-v0_8-whitegrid')
plt .rcParams .update ({'figure.dpi':120 ,'axes.spines.top':False ,'axes.spines.right':False })
pd .set_option ('display.max_columns',30 )
print ('\u9879\u76ee\u6839\u76ee\u5f55:',PROJECT_ROOT )
print ('RQ2 \u76ee\u5f55:',RQ2 )

## Analysis stage 3

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
required_files ={
'\u6240\u6709 case \u7684\u533a\u57df\u6807\u7b7e':DATA /'forecast_case_regimes.csv',
'\u533a\u57df\u6837\u672c\u6c47\u603b':DATA /'regime_summary.csv',
'\u516d\u6a21\u578b × \u56db\u533a\u57df\u6c47\u603b':DATA /'rq2_regime_model_summary.csv',
'\u6309 drifter bootstrap':DATA /'rq2_cluster_bootstrap_summary.csv',
'\u533a\u57df\u5dee\u5f02 bootstrap':DATA /'rq2_cluster_bootstrap_regime_contrasts.csv',
'FNN \u914d\u5bf9\u4f18\u52bf':DATA /'rq2_model_advantage_by_regime.csv',
'\u9608\u503c\u654f\u611f\u6027':DATA /'rq2_sensitivity_core_shelf_contrasts.csv',
'1–72 h \u533a\u57df\u8bef\u5dee':DATA /'regional_lead_time'/'six_model_regional_lead_summary.csv',
'\u9ed1\u6f6e\u533a\u57df\u91cd\u8bad\u6269\u5c55':DATA /'kuroshio_all_models'/'kuroshio_global_vs_regional_all_models.csv',
}
file_audit =pd .DataFrame ([
{'\u5185\u5bb9':name ,'\u5b58\u5728':path .exists (),'\u76f8\u5bf9\u8def\u5f84':str (path .relative_to (PROJECT_ROOT ))}
for name ,path in required_files .items ()
])
display (file_audit )

## Analysis stage 4

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
cases =pd .read_csv (DATA /'forecast_case_regimes.csv')
splits =pd .read_csv (PROCESSED /'drifter_splits.csv')

assigned =splits .groupby ('partition')['ID'].nunique ().rename ('assigned_drifters')
usable =cases .groupby ('partition')['ID'].nunique ().rename ('drifters_with_usable_cases')
n_cases =cases .groupby ('partition').size ().rename ('forecast_cases')
split_table =pd .concat ([assigned ,usable ,n_cases ],axis =1 ).reindex (['training','calibration','test'])
split_table .loc ['Total']=split_table .sum ()
display (split_table .astype (int ))

# Implementation note translated in the organised English notebook.
leakage_check =cases .groupby ('ID')['partition'].nunique ().max ()
print ('\u540c\u4e00 drifter \u51fa\u73b0\u7684\u6700\u5927 partition \u6570:',leakage_check )
assert leakage_check ==1 
assert len (cases )==5117 

## Analysis stage 5

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
env_cols =['depth_m','current_speed_clim','distance_to_kuroshio_km']
display (cases [env_cols ].describe ().T .round (3 ))

fig_path =FIG /'glorys_surface_currents.png'
if fig_path .exists ():
    display (Markdown ('**GLORYS \u957f\u671f\u5e73\u5747\u8868\u5c42\u6d41\u56fe：** \u989c\u8272\u8868\u793a\u6d41\u901f\u5927\u5c0f，\u7bad\u5934\u8868\u793a\u957f\u671f\u5e73\u5747\u65b9\u5411。'))
    display (Image (filename =str (fig_path ),width =850 ))

## Analysis stage 6

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
regime_order =['Shelf','Kuroshio Core','Transition','Other Offshore']
regime_summary =pd .read_csv (DATA /'regime_summary.csv')
test_summary =(regime_summary [regime_summary ['partition'].eq ('test')]
.set_index ('regime').reindex (regime_order ))
display (test_summary .round (3 ))

map_path =FIG /'test_ocean_regimes.png'
if map_path .exists ():
    display (Image (filename =str (map_path ),width =850 ))

print ('Test cases \u603b\u6570:',int (test_summary ['n_cases'].sum ()))
print ('\u6ce8\u610f：\u5404\u533a\u57df\u7684 drifter \u6570\u4e0d\u80fd\u76f8\u52a0，\u56e0\u4e3a\u540c\u4e00 test drifter \u53ef\u5728\u4e0d\u540c\u65f6\u523b\u7ecf\u8fc7\u591a\u4e2a\u533a\u57df。')

## Analysis stage 7

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
summary = pd.read_csv(DATA / 'rq2_regime_model_summary.csv')
model_order = ['Persistence', 'Mean Flow', 'VAR(1)', 'Ridge', 'KNN', 'FNN']

median_table = (summary.pivot(index='model', columns='regime', values='median_72h_km')
                .reindex(index=model_order, columns=regime_order))
rmse_table = (summary.pivot(index='model', columns='regime', values='rmse_72h_km')
              .reindex(index=model_order, columns=regime_order))
display(Markdown('### Median 72 h endpoint error (km; lower is better)'))
display(median_table.round(2))
display(Markdown('### RMSE 72 h endpoint error (km; lower is better)'))
display(rmse_table.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, table, title in zip(axes, [median_table, rmse_table], ['Median 72 h error', '72 h RMSE']):
    table.T.plot(kind='bar', ax=ax, width=0.82)
    ax.set_title(title)
    ax.set_xlabel('Ocean regime')
    ax.set_ylabel('Error (km)')
    ax.tick_params(axis='x', rotation=18)
axes[0].legend_.remove()
axes[1].legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Analysis stage 8

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
lead_file =DATA /'regional_lead_time'/'six_model_regional_lead_summary.csv'
if lead_file .exists ():
    lead =pd .read_csv (lead_file )
    fig ,axes =plt .subplots (2 ,2 ,figsize =(13 ,9 ),sharex =True )
    for ax ,regime in zip (axes .flat ,regime_order ):
        part =lead [lead ['regime'].eq (regime )]
        for model in model_order :
            m =part [part ['model'].eq (model )].sort_values ('lead_time_hours')
            ax .plot (m ['lead_time_hours'],m ['rmse_km'],lw =2 ,label =model )
        ax .set_title (regime )
        ax .set_ylabel ('RMSE (km)')
        ax .set_xlabel ('Lead time (h)')
    handles ,labels =axes [0 ,0 ].get_legend_handles_labels ()
    fig .legend (handles ,labels ,loc ='upper center',ncol =6 ,frameon =False )
    fig .suptitle ('Global models evaluated within each regime',y =1.01 ,fontweight ='bold')
    plt .tight_layout ()
    plt .show ()
else :
    print ('\u7f3a\u5c11 hourly summary；\u53ef\u8fd0\u884c RQ2/20_regional_lead_time_curves.py \u751f\u6210。')

## Analysis stage 9

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
boot =pd .read_csv (DATA /'rq2_cluster_bootstrap_summary.csv')
contrast =pd .read_csv (DATA /'rq2_cluster_bootstrap_regime_contrasts.csv')

fnn_ci =boot [boot ['model'].eq ('FNN')][
['regime','n_cases','n_drifters','median_72h_km','median_ci_lower_km',
'median_ci_upper_km','rmse_72h_km','rmse_ci_lower_km','rmse_ci_upper_km']
].set_index ('regime').reindex (regime_order )
display (Markdown ('### FNN \u5728\u56db\u4e2a\u533a\u57df\u7684 cluster bootstrap interval'))
display (fnn_ci .round (2 ))

core_shelf =contrast [(contrast ['regime_a'].eq ('Kuroshio Core'))&(contrast ['regime_b'].eq ('Shelf'))]
display (Markdown ('### Kuroshio Core − Shelf（\u6b63\u6570\u8868\u793a Core \u8bef\u5dee\u66f4\u5927）'))
display (core_shelf [['model','metric','difference_km','ci_lower_km','ci_upper_km','ci_excludes_zero']].round (2 ))

## Analysis stage 10

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
adv = pd.read_csv(DATA / 'rq2_model_advantage_by_regime.csv')
adv_table = adv.pivot(index='comparator', columns='regime', values='mean_advantage_km').reindex(columns=regime_order)
display(Markdown('### Mean paired advantage: comparator error − FNN error (km)'))
display(adv_table.round(2))

fig_path = FIG / '17_fig3_fnn_paired_advantage.png'
if fig_path.exists():
    display(Image(filename=str(fig_path), width=850))

## Analysis stage 11

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
sens = pd.read_csv(DATA / 'rq2_sensitivity_core_shelf_contrasts.csv')
stability = (sens.groupby(['model', 'metric'])
             .agg(settings=('core_minus_shelf_km', 'size'),
                  positive_settings=('core_minus_shelf_km', lambda x: int((x > 0).sum())),
                  significant_settings=('ci_excludes_zero', 'sum'),
                  min_difference_km=('core_minus_shelf_km', 'min'),
                  max_difference_km=('core_minus_shelf_km', 'max'))
             .reset_index())
display(stability.round(2))

fig_path = FIG / '17_fig4_core_shelf_threshold_sensitivity.png'
if fig_path.exists():
    display(Image(filename=str(fig_path), width=850))

## Analysis stage 12

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
ext_file =DATA /'kuroshio_all_models'/'kuroshio_global_vs_regional_all_models.csv'
if ext_file .exists ():
    ext =pd .read_csv (ext_file )
    show_cols =['model','n_cases','n_drifters','global_rmse_72h_km','rmse_72h_km',
    'rmse_improvement_pct','global_median_72h_error_km',
    'median_72h_error_km','median_improvement_pct']
    display (ext [show_cols ].round (2 ))

    x =np .arange (len (ext ))
    width =0.36 
    fig ,ax =plt .subplots (figsize =(10 ,4.8 ))
    ax .bar (x -width /2 ,ext ['global_rmse_72h_km'],width ,label ='Global training')
    ax .bar (x +width /2 ,ext ['rmse_72h_km'],width ,label ='Kuroshio-only training')
    ax .set_xticks (x ,ext ['model'].str .replace ('Regional ','',regex =False ),rotation =12 )
    ax .set_ylabel ('72 h RMSE (km)')
    ax .set_title ('Same Kuroshio test cases; different training data')
    ax .legend (frameon =False )
    plt .tight_layout ()
    plt .show ()
else :
    print ('\u672a\u627e\u5230 Kuroshio-only \u91cd\u8bad\u7ed3\u679c。')

## Analysis stage 13

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 14

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.